# Approach: Min-K% and XGBoost
In this approach, I use the same pipeline as in Milestone 1 to load the data, extract features, and process in batches. However, some additions are made, including a new min-k% feature (members will tend to have a higher model confidence on the most surprising, least likely tokens) and an XGBoost classifier to see if better performance is possible. Cross-model difference and ratio features were added, since, after some examination feature importance, it was clear that the 360m model's feature were adding more to the classifier.    
Note: I also tried to use ReCaLL, however, this feature did not end up contributing to the overall performance of the classifier models, but rather, added unnecessary noise. I've taken this feature out retroactively, but this was among the features extracted initially.

In [ ]:
import sys
# Path to the model5 directory after uploading to Colab.
# If you uploaded to /content/, this is correct as-is.
sys.path.insert(0, '/content/model5')

import pickle
import warnings
warnings.filterwarnings('ignore')

import torch

from models import load_data, load_models_and_tokenizer
from batches import process_all_splits, save_features, load_features
from build_membership_classifier import (
    build_membership_classifier, compute_zlib_baseline, generate_submission,
)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
SAVE_DIR = "/content/drive/MyDrive/mia_features/"

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Always run this — splits are needed even when loading cached features.
splits = load_data()

In [ ]:
# Skip this cell if loading from cached features.
tokenizer, model_135m, model_360m, ref_model_135m, ref_model_360m = load_models_and_tokenizer(device)

In [ ]:
features = process_all_splits(
    model_135m, model_360m, ref_model_135m, ref_model_360m,
    tokenizer, splits, batch_size=500,
)

In [ ]:
save_features(SAVE_DIR, features)

In [ ]:
# Run this cell instead of the extraction cells when resuming a session with saved features.
features = load_features(SAVE_DIR)

In [ ]:
results = build_membership_classifier(
    features["train_135m"], features["train_360m"],
    features["val_135m"],   features["val_360m"],
)

In [ ]:
# Results summary
print("MEMBERSHIP INFERENCE ATTACK RESULTS")
print("=" * 50)

zlib_result = compute_zlib_baseline(splits["validation"])

for name, r in results.items():
    print(f"\n{name}:  AUC={r['auc']:.4f}  TPR@FPR=0.1={r['tpr_at_fpr_01']:.4f}  Acc={r['accuracy']:.4f}")

best_name = max(results, key=lambda x: results[x]["auc"])
best      = results[best_name]
print(f"\nBest: {best_name}  AUC={best['auc']:.4f}  TPR@FPR=0.1={best['tpr_at_fpr_01']:.4f}")

# Save classifier to Google Drive
classifier_path = SAVE_DIR + "membership_classifier.pkl"
with open(classifier_path, "wb") as f:
    pickle.dump(best, f)
print(f"Saved classifier to {classifier_path}")

# Generate Kaggle submission — saved locally so files.download can pick it up
submission_path = "membership_inference_submission.csv"
submission_df = generate_submission(
    features["test_135m"], features["test_360m"],
    best, splits["test"],
    output_path=submission_path,
)
print(submission_df.head())

In [35]:
from google.colab import files
files.download('membership_inference_submission.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>